# **Utility Fuctions**

In [2]:
import numpy as np
import cv2 as cv


def countFrames(path):
    video=cv.VideoCapture(path)
    count=0
    while True:
        ret,frame=video.read()
        if not ret: break
        count=count+1

    video.release()
    return count


def sneakPeekVideo(frames):
    for i in range(16):
        cv.imshow('frame', frames[i])
        cv.waitKey(500)  
    cv.destroyAllWindows()
    


def loadClip(path):
    video=cv.VideoCapture(path)
    # count the total number of frames in the video
    count=countFrames(path)

    # generate 16 even spacaed numbers , btw 0 and the max number of frames
    idxs = np.linspace(0, count-1, 16).astype(int)

    # extract those frames
    frames=[]
    for i in idxs:
        video.set(cv.CAP_PROP_POS_FRAMES, i)
        ret,frame=video.read()
        if ret:
            # processing the frame before appening
            frame=cv.resize(frame,(112,112))
            frame=cv.cvtColor(frame,cv.COLOR_BGR2RGB)
            frame=frame.astype("float32")/255.0

            frames.append(frame)

    video.release()
    return np.stack(frames)
    

In [3]:
import random

def augment(frames):
    # frames: numpy (16, 112, 112, 3) float32, RGB, 0-1

    # 1. horizontal flip (50% chance)
    if random.random() < 0.5:
        frames = frames[:, :, ::-1, :].copy()

    # 2. random brightness / contrast jitter
    if random.random() < 0.5:
        gain = random.uniform(0.7, 1.3)
        frames = np.clip(frames * gain, 0, 1)

    # 3. random crop: take a random 96x96 window, resize back to 112x112
    if random.random() < 0.5:
        h, w = 96, 96
        top = random.randint(0, 112 - h)
        left = random.randint(0, 112 - w)
        cropped = frames[:, top:top+h, left:left+w, :]
        frames = np.array([cv.resize(f, (112, 112)) for f in cropped])

    return frames

In [4]:
import torch, glob
import os



class ActionDataset(torch.utils.data.Dataset):
    def __init__(self, root, augment_data=True):
        self.classes = {"wave":0, "walk":1, "sitting":2, "standing":3}
        self.augment_data = augment_data
        self.items = []
        paths = glob.glob(root + "/*/*.mp4")
        for path in paths:
            names = path.split(os.sep)
            label = self.classes[names[-2]]
            video = loadClip(path)                    # numpy, stays numpy
            self.items.append((video, label))

    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        video, label = self.items[i]
        if self.augment_data:
            video = augment(video)                    # fresh random every call
        return torch.from_numpy(video), label
    


# **3D Neural Network**

In [5]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv3d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Flatten(),
            nn.Linear(25088, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
        )

    def forward(self, x):
        x = x.permute(0, 4, 1, 2, 3)
        return self.layers(x)

In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| GPU count:", torch.cuda.device_count())

Using device: cuda | GPU count: 2


In [7]:
trainDs = ActionDataset("/kaggle/input/datasets/zadifmustafa/trainingdata/data", augment_data=True)

loader = torch.utils.data.DataLoader(trainDs, batch_size=4, shuffle=True)
testDs = ActionDataset("/kaggle/input/datasets/zadifmustafa/testingdata/test", augment_data=False)
testLoader = torch.utils.data.DataLoader(testDs, batch_size=4, shuffle=False)



In [8]:
print(len(trainDs))    # expect 62
print(len(testDs))     # expect 20

66
16


# **Testing**

In [9]:
import numpy as np
from sklearn.metrics import confusion_matrix
import torch

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for clips, labels in loader:
            clips = clips.to(device)

            out = model(clips)
            preds = torch.argmax(out, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    matrix = confusion_matrix(all_labels, all_preds)
    accuracy = matrix.trace() / matrix.sum()
    return accuracy, matrix

# **Training**

In [12]:
import torch, glob
import os
# Replace `model = CNN().to(device)` with:
model = CNN()
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(50):
    model.train()
    for clips, labels in loader:
        clips = clips.to(device)
        labels = labels.to(device)

        out = model(clips)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc, matrix = evaluate(model, testLoader)
    print(f"epoch {epoch}, loss {loss.item():.4f}, test accuracy {acc:.3f}")

epoch 0, loss 1.3427, test accuracy 0.250
epoch 1, loss 1.3495, test accuracy 0.375
epoch 2, loss 1.3892, test accuracy 0.250
epoch 3, loss 1.4035, test accuracy 0.250
epoch 4, loss 1.3434, test accuracy 0.312
epoch 5, loss 1.5620, test accuracy 0.250
epoch 6, loss 1.2714, test accuracy 0.438
epoch 7, loss 1.1250, test accuracy 0.250
epoch 8, loss 0.8973, test accuracy 0.562
epoch 9, loss 0.4932, test accuracy 0.688
epoch 10, loss 0.2366, test accuracy 0.625
epoch 11, loss 0.7260, test accuracy 0.750
epoch 12, loss 0.0943, test accuracy 0.812
epoch 13, loss 0.3187, test accuracy 0.688
epoch 14, loss 0.1192, test accuracy 0.688
epoch 15, loss 0.4596, test accuracy 0.812
epoch 16, loss 0.4923, test accuracy 0.688
epoch 17, loss 0.3112, test accuracy 0.688
epoch 18, loss 0.0828, test accuracy 0.750
epoch 19, loss 1.1785, test accuracy 0.688
epoch 20, loss 0.3690, test accuracy 0.812
epoch 21, loss 0.1396, test accuracy 0.875
epoch 22, loss 0.4686, test accuracy 0.875
epoch 23, loss 0.2013

In [11]:
train_acc, train_matrix = evaluate(model, loader)
test_acc,  test_matrix  = evaluate(model, testLoader)
print(f"train: {train_acc:.3f}   test: {test_acc:.3f}")

# the leak test
train_paths = set(p for p, _ in trainDs.items)
test_paths  = set(p for p, _ in testDs.items)
print("overlap:", len(train_paths & test_paths))

# and the class balance of each set
from collections import Counter
print("train classes:", Counter(l for _, l in trainDs.items))
print("test  classes:", Counter(l for _, l in testDs.items))

train: 1.000   test: 0.875


TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
torch.save(model.state_dict(), "cnn_model.pth")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get predictions on the test dataset
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for clips, labels in testLoader:
        clips = clips.to(device)
        outputs = model(clips)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Your notebook's class-to-label mapping
class_names = ["wave", "walk", "sitting", "standing"]

# Calculate confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3])

# Plot it
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("3D CNN - Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [15]:
train_acc, _ = evaluate(model, loader)
print("train acc:", train_acc)
print("params:", sum(p.numel() for p in model.parameters()))
torch.save(model.state_dict(), "cnn_model.pth")

train acc: 1.0
params: 3282436
